<a href="https://colab.research.google.com/github/Sriyansh-36-AI-NITJ/Deep-Learning-Lab/blob/main/Classify_Bing_Queries_End_Sem_Practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 60.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import pandas as pd
import spacy

# Load the spaCy English model
print("Loading NLP model...")
nlp = spacy.load("en_core_web_sm")
print("Model loaded successfully!")

Loading NLP model...
Model loaded successfully!


In [3]:
def classify_query(query):
    """
    Classifies a Bing search query as 'Specific' or 'Generic'.
    """
    # Handle empty/null queries
    if not isinstance(query, str):
        return "Generic"

    query_lower = query.lower()

    # 1. Fast Rule-Based Check for explicit local intent
    local_keywords = ["near me", "nearby", "local", "zip code"]
    if any(keyword in query_lower for keyword in local_keywords):
        return "Specific"

    # 2. Named Entity Recognition (NER)
    doc = nlp(query)

    # Check for Geopolitical Entities, Locations, Organizations, or Facilities
    specific_entity_labels = {"GPE", "LOC", "ORG", "FAC"}

    for ent in doc.ents:
        if ent.label_ in specific_entity_labels:
            return "Specific"

    # Default to generic if no specific entities or rules apply
    return "Generic"

In [4]:
# URL for the raw TSV file from the Bing dataset (Jan 2020 by Country as an example)
raw_url = "https://raw.githubusercontent.com/microsoft/BingCoronavirusQuerySet/master/data/2020/QueriesByCountry_2020-01-01_2020-01-31.tsv"

print("Downloading and loading dataset...")
# Load the dataset using pandas, specifying the tab separator
df = pd.read_csv(raw_url, sep='\t')

# Fill empty fields just in case
df.fillna("Unknown", inplace=True)

# Display the top 5 rows to confirm it loaded
df.head()

,Date,Query,IsImplicitIntent,Country,PopularityScore
0,2020-01-01,kalitta air,True,United States,1
1,2020-01-01,sars virus,True,United States,15
2,2020-01-01,auswärtiges amt,True,Germany,100
3,2020-01-01,arrowe park hospital,True,United Kingdom,1
4,2020-01-01,冠状病毒,False,China,1


In [5]:
print("Classifying queries... (this may take a minute depending on data size)")

# Create a new column 'Classification' by applying our function to the 'Query' column
df['Classification'] = df['Query'].apply(classify_query)

print("Classification complete!")

Classifying queries... (this may take a minute depending on data size)
Classification complete!


In [6]:
# Look at a sample of queries and how they were classified
print(df[['Query', 'Classification']].sample(15))

# Save the updated dataframe to a new TSV file
output_filename = "Classified_BingQueries.tsv"
df.to_csv(output_filename, sep='\t', index=False)

print(f"\nData successfully saved to {output_filename}")

                                                 Query Classification
4958                          sintomas del coronavirus       Specific
1499                            2019 novel coronavirus        Generic
10183                           coronavirus gefährlich        Generic
26056                              corona virus update       Specific
17436                                corona virus name       Specific
3384                                quebec coronavirus        Generic
9490                   where did the coronavirus start        Generic
26306                                     virus corona        Generic
9876                                 coronavirus india       Specific
27519                        sintomas del corona virus       Specific
30491                 coronavirus at elmhurst hospital        Generic
28212  how many people have died from the corona virus        Generic
30122                 department of health coronavirus       Specific
13467               